# Exercise — Trace Lineage & Specify Monitoring

**Trailhead Provisions** is pre-instrumented with lineage. Trace the blast radius of a
customer-data breach, confirm the regulated report's sources, and specify the monitoring that
would surface a breach early. See `INSTRUCTIONS.md`.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import GovernedCatalog

gc = GovernedCatalog("trailhead.db")
display(gc.lineage.events())
print("Datasets:", sorted(gc.lineage.datasets()))

## 1. Trace impact
Trace everything **downstream of `customer`** (a PII breach blast radius) and the **upstream sources** of `quarterly_sustainability` (the CSRD report).

In [ ]:
pii_impact = gc.lineage.trace_downstream("customer")
print("Downstream of a customer breach:", pii_impact)
report_sources = gc.lineage.trace_upstream("quarterly_sustainability")
print("Upstream sources of the CSRD report:", report_sources)

## 2. Monitoring specification
Replace the cell below with your spec.

### Monitoring specification

| Signal | Source dataset/job | Threshold | Owner | Ties to |
|---|---|---|---|---|
| Carbon unit out of range | `build_order_carbon` | any `carbon_kg` > 100 | Inventory/Fulfillment | accuracy SLO; CSRD report |
| Consent mismatch rate | `build_customer_360` | > 1% loyalty≠marketing | Customer Identity | consistency SLO; GDPR |
| Freshness of `quarterly_sustainability` | report job | > 24h since last run | CSRD Reporting Owner | report SLA |
| Null mandatory attrs (`product.category`) | product producer | > 0% | Inventory/Fulfillment | completeness SLO |
| Lineage event missing | any job | run with no lineage event | Platform Engineering | governance audit |

**Breach reading:** a `customer` breach exposes `customer_360` and `marketing_audience`
downstream. A `shipment` change reaches `quarterly_sustainability` with no intervening gate.
**Gap the current signals miss:** there is no monitor on the `customer_id`↔`cust_ref` join
quality, so a resolution regression wouldn't alarm — add match-rate monitoring on
`build_customer_360`.